# Pandas — Structured Data in Python

Pandas is the primary library for working with tabular data. It sits on top of NumPy and adds labelled rows/columns, missing value handling, and a rich API for data wrangling.

Topics:
- Series and DataFrame creation
- Inspection: info, describe, head/tail
- Selecting: `[]`, `.loc`, `.iloc`
- Filtering with conditions
- Adding, renaming, and dropping columns
- Handling missing values
- Value counts, sorting, applying functions
- String operations (`.str` accessor)
- Datetime handling

In [ ]:
import pandas as pd
import numpy as np
print(pd.__version__)

## 1. Series

A `Series` is a 1-D labelled array — think: a single column of a spreadsheet.

In [ ]:
# Creating a Series
s = pd.Series([10, 20, 30, 40, 50])
print(s)
print()

# Custom index
s_named = pd.Series([85, 92, 78], index=['Alice', 'Bob', 'Carol'])
print(s_named)
print('Alice\'s score:', s_named['Alice'])

In [ ]:
# Series operations — element-wise, like NumPy
temps = pd.Series({'Mon':22, 'Tue':25, 'Wed':19, 'Thu':28, 'Fri':30})
print(temps * 9/5 + 32)      # celsius to fahrenheit
print(temps[temps > 24])     # boolean filter
print(temps.mean(), temps.max(), temps.idxmax())

## 2. DataFrame

A `DataFrame` is a 2-D table with labelled rows (index) and columns. The workhorse of data science.

In [ ]:
# Creating a DataFrame
data = {
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age':    [25, 30, 22, 35, 28],
    'city':   ['NYC', 'LA', 'NYC', 'Chicago', 'LA'],
    'salary': [70000, 85000, 60000, 95000, 75000],
    'score':  [88.5, 92.0, 79.5, 95.0, 83.5]
}

df = pd.DataFrame(data)
print(df)

In [ ]:
# Inspection
print('Shape:', df.shape)        # (rows, cols)
print('Columns:', df.columns.tolist())
print('Index:', df.index.tolist())
print()
df.info()

In [ ]:
print(df.describe())     # stats for numeric columns
print()
print(df.describe(include='all'))  # includes strings

In [ ]:
print(df.head(3))   # first 3 rows
print()
print(df.tail(2))   # last 2 rows
print()
print(df.sample(3, random_state=42))  # 3 random rows

## 3. Selecting Data

| Method | Use for |
|--------|--------|
| `df['col']` | Single column (returns Series) |
| `df[['a','b']]` | Multiple columns (returns DataFrame) |
| `df.loc[row, col]` | Label-based — use index labels and column names |
| `df.iloc[row, col]` | Position-based — use integer positions |

In [ ]:
# Column selection
print(df['name'])           # single column → Series
print()
print(df[['name','salary']]) # multiple columns → DataFrame

In [ ]:
# .loc — label based
print(df.loc[0])             # row at index label 0
print()
print(df.loc[0:2, 'name':'city'])  # rows 0-2, cols name to city (inclusive!)
print()
print(df.loc[df['age'] > 25])      # boolean filter with .loc

In [ ]:
# .iloc — position based
print(df.iloc[0])           # first row
print()
print(df.iloc[0:3, 0:2])    # rows 0-2 (exclusive!), cols 0-1
print()
print(df.iloc[-1])          # last row

## 4. Filtering (Boolean Indexing)

In [ ]:
# Single condition
print(df[df['city'] == 'NYC'])
print()
# Multiple conditions — use & | ~ (NOT 'and' 'or' 'not')
print(df[(df['age'] > 25) & (df['salary'] > 80000)])
print()
# isin — check against a list
print(df[df['city'].isin(['NYC', 'LA'])])
print()
# between — numeric range
print(df[df['salary'].between(70000, 90000)])

In [ ]:
# query() — cleaner syntax for filtering
print(df.query('age > 25 and city == "LA"'))
print()
# Use @ to reference Python variables inside query
min_salary = 75000
print(df.query('salary >= @min_salary'))

## 5. Adding, Renaming, and Dropping Columns

In [ ]:
df2 = df.copy()

# Add new column
df2['bonus'] = df2['salary'] * 0.10
df2['grade'] = df2['score'].apply(lambda s: 'A' if s >= 90 else 'B' if s >= 80 else 'C')
print(df2[['name','salary','bonus','grade']])

# Rename columns
df2.rename(columns={'salary': 'annual_salary', 'score': 'test_score'}, inplace=True)
print(df2.columns.tolist())

# Drop columns
df2.drop(columns=['bonus'], inplace=True)
print(df2.columns.tolist())

# Drop rows
df2.drop(index=[0, 2], inplace=True)
print(df2)

## 6. Handling Missing Values

In [ ]:
# Create a DataFrame with missing values
df_missing = pd.DataFrame({
    'name':  ['Alice', 'Bob', None, 'Dave', 'Eve'],
    'age':   [25, np.nan, 22, np.nan, 28],
    'score': [88.5, 92.0, np.nan, 95.0, np.nan]
})
print(df_missing)
print()
print('Missing per column:')
print(df_missing.isna().sum())
print()
print('Missing %:')
print(df_missing.isna().mean() * 100)

In [ ]:
# Drop rows with any NaN
print(df_missing.dropna())
print()
# Drop rows only if ALL values are NaN
print(df_missing.dropna(how='all'))
print()
# Drop columns with more than 1 NaN
print(df_missing.dropna(axis=1, thresh=4))

# Fill missing values
df_filled = df_missing.copy()
df_filled['age'].fillna(df_filled['age'].median(), inplace=True)   # fill with median
df_filled['score'].fillna(df_filled['score'].mean(), inplace=True)  # fill with mean
df_filled['name'].fillna('Unknown', inplace=True)
print(df_filled)

## 7. Sorting, Value Counts, and Unique Values

In [ ]:
print(df.sort_values('salary', ascending=False))   # sort by salary descending
print()
print(df.sort_values(['city','age']))               # multi-column sort
print()
print('City counts:\n', df['city'].value_counts())
print('Unique cities:', df['city'].unique())
print('N unique:', df['city'].nunique())
print()
print(df[['city','salary']].value_counts())

## 8. Applying Functions

In [ ]:
df3 = df.copy()

# apply on a column (Series)
df3['salary_k'] = df3['salary'].apply(lambda x: f'${x/1000:.0f}k')
print(df3[['name','salary','salary_k']])

# apply on a DataFrame (row-wise with axis=1)
df3['seniority'] = df3.apply(
    lambda row: 'Senior' if row['age'] > 28 and row['salary'] > 80000 else 'Junior',
    axis=1
)
print(df3[['name','age','salary','seniority']])

# map — element-wise transformation using a dict
city_map = {'NYC': 'East', 'LA': 'West', 'Chicago': 'Central'}
df3['region'] = df3['city'].map(city_map)
print(df3[['name','city','region']])

## 9. String Operations (`.str` accessor)

In [ ]:
emails = pd.Series(['alice@example.com', 'BOB@GMAIL.COM', ' carol@co.uk ', 'dave'])

print(emails.str.lower())             # lowercase
print(emails.str.strip())             # remove whitespace
print(emails.str.contains('@'))       # boolean — contains?
print(emails.str.split('@'))          # split into list
print(emails.str.extract(r'@(.*)\.'))  # regex extract domain
print(emails.str.startswith('alice'))

## 10. Datetime Handling (`.dt` accessor)

In [ ]:
dates = pd.to_datetime(['2024-01-15', '2024-06-30', '2025-03-08'])
s = pd.Series(dates)

print(s.dt.year)
print(s.dt.month)
print(s.dt.day)
print(s.dt.day_name())     # Monday, Tuesday, ...
print(s.dt.quarter)
print(s.dt.dayofweek)      # 0=Monday, 6=Sunday

# Time difference
today = pd.Timestamp('2025-05-05')
print((today - s).dt.days)   # days since each date

## Quick Summary

| Task | Code |
|------|------|
| Select column | `df['col']` or `df.col` |
| Select rows | `df.loc[label]` or `df.iloc[idx]` |
| Filter | `df[df['col'] > val]` or `df.query('...')` |
| Add column | `df['new'] = ...` |
| Drop | `df.drop(columns=['a'])` |
| Missing | `.isna()`, `.fillna()`, `.dropna()` |
| Sort | `.sort_values('col')` |
| Count | `.value_counts()` |
| Transform | `.apply(func)`, `.map(dict)` |
| Strings | `.str.lower()`, `.str.contains()` |
| Dates | `.dt.year`, `.dt.month`, `.dt.day_name()` |

**Next →** [03 – Data Manipulation](../03-data-manipulation/)